# 08: Visual Comparison of Pointcloud Labeling Methods

This notebook provides a modular and reproducible framework to visually compare different strategies for projecting 2D segmentation masks onto a 3D pointcloud, using Project Aria data.

### In this notebook, we will:
1. Data loading and setup
2. Mask extraction utilities (SAM2)
3. Baseline projection: single frame, single camera
4. Accumulated projection: different frames combination
5. Multi-camera projection: different camera combination
6. Final comparison: best of 4. vs best of 5.

As a sample, we will use `kettle_and_forklift_recording.vrs` located in the local `data/raw/kettle_and_forklift/` directory and its corresponding segmentation masks located in `data/outputs/segmentation/kettle_and_forklift/sam2/masks` .


## 8.1 Data Loading and Setup

In this section, we load the 3D pointcloud and camera trajectory data produced by MPS

In [1]:
# Import required libraries and load pointcloud and trajectory data
import os
import pandas as pd
from aria_pylib import find_points_file, find_trajectory_file

# Define the directory containing MPS outputs
MPS_DIR = os.path.join('..', 'data', 'raw', 'kettle_and_forklift', 'mps_kettle_and_forklift_recording_vrs', 'slam')

# Load the pointcloud file
points_path = find_points_file(MPS_DIR)
print(f"Loading pointcloud from: {points_path}")
points_df = pd.read_csv(points_path)

# Load trajectory file
trajectory_path = find_trajectory_file(MPS_DIR)
print(f"Loading trajectory from: {trajectory_path}")
trajectory_df = pd.read_csv(trajectory_path)
print(f"Loaded {len(trajectory_df)} poses.")

Loading pointcloud from: ..\data\raw\kettle_and_forklift\mps_kettle_and_forklift_recording_vrs\slam\semidense_points.csv.gz
Loading trajectory from: ..\data\raw\kettle_and_forklift\mps_kettle_and_forklift_recording_vrs\slam\closed_loop_trajectory.csv
Loaded 50626 poses.


## 8.2 Mask Extraction Utilities

This section provides utility functions to extract 2D segmentation masks from the dataset, given a specific timestamp and camera. These functions will be used to retrieve the masks needed for each projection experiment.

In [2]:
# Utility function to load a segmentation mask for a given frame and camera
import cv2

def load_mask(masks_dir, frame_idx, camera_name="camera-rgb"):
    """
    Loads a 2D segmentation mask for the specified frame and camera.
    """
    mask_filename = f"{frame_idx:06d}.png"
    mask_path = os.path.join(masks_dir, camera_name, mask_filename) if os.path.isdir(os.path.join(masks_dir, camera_name)) else os.path.join(masks_dir, mask_filename)
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if mask is None:
        raise FileNotFoundError(f"Could not load mask at {mask_path}. Check if the file exists!")
    print(f"Loaded mask from {mask_path}\nShape: {mask.shape}")
    return mask

# example
masks_dir = os.path.join('..', 'data', 'outputs', 'segmentation', 'kettle_and_forklift', 'sam2', 'masks')
frame_idx = 50
mask = load_mask(masks_dir, frame_idx)

Loaded mask from ..\data\outputs\segmentation\kettle_and_forklift\sam2\masks\000050.png
Shape: (1408, 1408)


## 8.3.1 Baseline Projection: Single Frame, Single Camera

In this section, we implement the baseline projection method: projecting the 3D pointcloud onto a single 2D segmentation mask (from the RGB camera) at a specific frame. The resulting labeled pointcloud will be used as a reference for comparison with more advanced strategies.
We will use the frame 50 (video at 10 fps) as an example

In [29]:
import os
import cv2
import pandas as pd
import numpy as np
from tqdm import tqdm
from scipy.spatial.transform import Rotation as R
from projectaria_tools.core import data_provider

# Setup paths and parameters
vrs_path = os.path.join('..', 'data', 'raw', 'kettle_and_forklift', 'kettle_and_forklift_recording.vrs')
masks_dir = os.path.join('..', 'data', 'outputs', 'segmentation', 'kettle_and_forklift', 'sam2', 'masks')
baseline_frame_idx = 50 

# Load Calibration
provider = data_provider.create_vrs_data_provider(vrs_path)
device_calib = provider.get_device_calibration()
rgb_calib = device_calib.get_camera_calib("camera-rgb")
T_Device_Camera = rgb_calib.get_transform_device_camera()

# Helper Functions
def load_mask(masks_dir, frame_idx):
    """Loads a 2D segmentation mask from disk."""
    mask_path = os.path.join(masks_dir, f"{frame_idx:06d}.png")
    return cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

def get_pose_from_vrs_timestamp(provider, trajectory_df, frame_idx, seg_fps=10, vrs_fps=30, stream_name="camera-rgb"):
    """Extracts exact hardware timestamp and finds the closest trajectory pose."""
    # Scale index from segmentation fps to native VRS fps
    vrs_frame_idx = int(frame_idx * (vrs_fps / seg_fps))
    
    stream_id = provider.get_stream_id_from_label(stream_name)
    image_data = provider.get_image_data_by_index(stream_id, vrs_frame_idx)
    capture_time_ns = image_data[1].capture_timestamp_ns
    
    # Match nanoseconds with the correct trajectory column
    if 'tracking_timestamp_us' in trajectory_df.columns:
        trajectory_times_ns = trajectory_df['tracking_timestamp_us'].to_numpy() * 1000
    elif 'timestamp_ns' in trajectory_df.columns:
        trajectory_times_ns = trajectory_df['timestamp_ns'].to_numpy()
    elif 'utc_timestamp_ns' in trajectory_df.columns:
        trajectory_times_ns = trajectory_df['utc_timestamp_ns'].to_numpy()
    else:
        raise KeyError(f"Timestamp column not found. Available: {trajectory_df.columns.tolist()}")

    closest_traj_idx = (np.abs(trajectory_times_ns - capture_time_ns)).argmin()
    return trajectory_df.iloc[closest_traj_idx]


def project_world_to_rgb_image(point_world, pose_row, rgb_calib, T_Device_Camera):
    """Projects a 3D point into 2D pixel coordinates."""
    quat = [pose_row['qx_world_device'], pose_row['qy_world_device'], pose_row['qz_world_device'], pose_row['qw_world_device']]
    rot_world_device = R.from_quat(quat).as_matrix()
    t_world_device = np.array([pose_row['tx_world_device'], pose_row['ty_world_device'], pose_row['tz_world_device']])
    
    point_device = rot_world_device.T @ (point_world - t_world_device)
    point_cam = T_Device_Camera.inverse() @ point_device
    return rgb_calib.project(point_cam)


def label_pointcloud_at_pose(points_df, pose_row, rgb_calib, T_Device_Camera, mask):
    """Assigns semantic labels to 3D points based on 2D mask projection."""
    labels, px, py, pz = [], [], [], []
    for _, row in tqdm(points_df.iterrows(), total=len(points_df), desc=f"Projecting points"):
        point_3d = np.array([row['px_world'], row['py_world'], row['pz_world']])
        pixel = project_world_to_rgb_image(point_3d, pose_row, rgb_calib, T_Device_Camera)
        label = 0
        if pixel is not None:
            u, v = int(round(pixel[0])), int(round(pixel[1]))
            if 0 <= v < mask.shape[0] and 0 <= u < mask.shape[1]:
                label = mask[v, u]
        labels.append(label)
        px.append(point_3d[0]); py.append(point_3d[1]); pz.append(point_3d[2])
    return pd.DataFrame({'px_world': px, 'py_world': py, 'pz_world': pz, 'semantic_label': labels})


print(f"Aligning mask for frame {baseline_frame_idx}...")
baseline_mask = load_mask(masks_dir, baseline_frame_idx)
baseline_pose = get_pose_from_vrs_timestamp(provider, trajectory_df, baseline_frame_idx)

print(f"Trajectory perfectly matched at index: {baseline_pose.name}")
labeled_baseline_df = label_pointcloud_at_pose(points_df, baseline_pose, rgb_calib, T_Device_Camera, baseline_mask)
print(f"Unique labels in 3D: {np.unique(labeled_baseline_df['semantic_label'])}")

Aligning mask for frame 50...
Trajectory perfectly matched at index: 4033


Projecting points: 100%|██████████| 239524/239524 [00:18<00:00, 12807.69it/s]


Unique labels in 3D: [0 1 2]


### 8.3.2 Visualize Baseline Projection in 3D
To verify the alignment, we visualize the labeled points in 3D using Rerun. We filter out the background (label 0) to exclusively inspect the points hit by the segmentation mask, along with the exact camera pose at the moment of capture.

In [31]:
import rerun as rr

# Filter out background to isolate target objects
valid_mask = labeled_baseline_df['semantic_label'] > 0
target_points = labeled_baseline_df[valid_mask]

points_xyz = target_points[['px_world', 'py_world', 'pz_world']].to_numpy(dtype=np.float32)
labels = target_points['semantic_label'].to_numpy()

# 1 = Red/Forklift, 2 = Cyan/Kettle
label_to_color = {1: [255, 0, 0], 2: [0, 255, 255]}
colors = np.array([label_to_color.get(int(l), [0, 255, 0]) for l in labels], dtype=np.uint8)

# Center the world around the trajectory for smooth navigation
traj_xyz = trajectory_df[['tx_world_device', 'ty_world_device', 'tz_world_device']].to_numpy(dtype=np.float32)
center = np.mean(traj_xyz, axis=0, keepdims=True)

points_xyz_centered = points_xyz - center
traj_xyz_centered = traj_xyz - center

# Camera position at capture time
camera_xyz = np.array([baseline_pose['tx_world_device'], baseline_pose['ty_world_device'], baseline_pose['tz_world_device']], dtype=np.float32)
camera_centered = camera_xyz - center

# Launch Rerun visualization
rr.init("Aria_Baseline_Frame_50", spawn=True)
rr.log("pointcloud/target_labels", rr.Points3D(points_xyz_centered, colors=colors, radii=0.0005))
rr.log("camera/trajectory", rr.LineStrips3D([traj_xyz_centered], colors=[[0,255,0]]))
rr.log("camera/pose_frame_50", rr.Points3D([camera_centered], colors=[[255,255,0]], radii=0.003))

print(f"Sent {len(points_xyz)} target points to Rerun viewer.")

Sent 41108 target points to Rerun viewer.


### 8.3.3 Save Baseline Pointcloud
We will now save the entire labeled pointcloud as a `.ply` file for external analysis

In [33]:
import os
import numpy as np
from plyfile import PlyData, PlyElement

print(f"Saving the entire labeled pointcloud to disk...")
pcd_dir = os.path.join('..', 'data', 'outputs', 'segmentation', 'kettle_and_forklift', 'pointclouds', 'multi_frame_comparison')
os.makedirs(pcd_dir, exist_ok=True)

# Extract ALL points and labels
all_xyz = labeled_baseline_df[['px_world', 'py_world', 'pz_world']].to_numpy(dtype=np.float32)
all_labels = labeled_baseline_df['semantic_label'].to_numpy()

save_label_to_color = {0: [180, 180, 180], 1: [255, 0, 0], 2: [0, 255, 255]}
all_colors = np.array([save_label_to_color.get(int(l), [180, 180, 180]) for l in all_labels], dtype=np.uint8)

# Compose structured array for ply format
vertex = np.empty(all_xyz.shape[0], dtype=[('x', 'f4'), ('y', 'f4'), ('z', 'f4'), ('red', 'u1'), ('green', 'u1'), ('blue', 'u1'), ('label', 'u1')])
vertex['x'] = all_xyz[:, 0]
vertex['y'] = all_xyz[:, 1]
vertex['z'] = all_xyz[:, 2]
vertex['red'] = all_colors[:, 0]
vertex['green'] = all_colors[:, 1]
vertex['blue'] = all_colors[:, 2]
vertex['label'] = all_labels.astype(np.uint8)

# Write to disk
ply = PlyData([PlyElement.describe(vertex, 'vertex')], text=True)
out_path = os.path.join(pcd_dir, f"pointcloud_frame_{baseline_frame_idx}.ply")
ply.write(out_path)

print(f"Full pointcloud saved successfully at: {out_path}")

Saving the entire labeled pointcloud to disk...
Full pointcloud saved successfully at: ..\data\outputs\segmentation\kettle_and_forklift\pointclouds\multi_frame_comparison\frame_50.ply


## 8.4 Multi-Frame Projection Comparison

In this section, we compare the effect of combining different sets of frames for the projection. We will accumulate the labels from multiple frames and analyze how the coverage and quality of the labeled pointcloud change as we add more frames.

We will test the following frame sets:
1) Only frame 50 (done previously in the baseline)
2) Frames 50 + 100 + 250
3) Frames 50 + 100 + 250 + 350 + 480
4) Frames 50 + 100 + 250 + 350 + 480 + 150 + 200

All the pointclouds will be saved as .ply files for external analysis but only the final comparison will be visualized in Rerun for qualitative analysis.

### 8.4.1 Labeling Accumulation from Multiple Frames
We will define an accumulation function that iterates over a given set of frames. For each frame, it fetches the exact hardware pose, projects the points, and updates the global labels (overwriting background points with newly discovered semantic labels).

In [34]:
from collections import OrderedDict
import numpy as np
import pandas as pd

# Multi-Frame Accumulation Helper
def accumulate_labels_multi_frame(points_df, trajectory_df, frame_indices, rgb_calib, T_Device_Camera, masks_dir, provider, mask_cache=None):
    """
    Projects and accumulates labels from multiple frames onto the 3D pointcloud.
    Uses exact hardware pose lookup for each frame to guarantee perfect alignment.
    """
    if mask_cache is None:
        mask_cache = {}
        
    # Start with all points labeled as 0 (background)
    labels_accum = np.zeros(len(points_df), dtype=int)
    
    for frame_idx in frame_indices:
        # Load mask (using cache to save I/O time)
        if frame_idx not in mask_cache:
            mask_cache[frame_idx] = load_mask(masks_dir, frame_idx)
        mask = mask_cache[frame_idx]
        
        # Get the exact hardware pose for this frame
        pose_row = get_pose_from_vrs_timestamp(provider, trajectory_df, frame_idx)
        
        # Project and label
        labeled_tmp = label_pointcloud_at_pose(points_df, pose_row, rgb_calib, T_Device_Camera, mask)
        
        # Accumulate labels: overwrite current 0s with new valid labels (>0)
        labels_accum = np.where(labeled_tmp['semantic_label'] > 0, labeled_tmp['semantic_label'], labels_accum)
        
    labeled_df = points_df.copy()
    labeled_df['semantic_label'] = labels_accum
    return labeled_df

# Define Frame Sets
frame_sets = [
    [50, 100, 250],
    [50, 100, 250, 350, 480],
    [50, 100, 250, 350, 480, 150, 200]
]

# Pre-load masks
all_frames = list(set([f for subset in frame_sets for f in subset]))
mask_cache = OrderedDict()
print("Pre-loading SAM2 masks to optimize execution...")
for f in all_frames:
    mask_cache[f] = load_mask(masks_dir, f)

# Execute Accumulation
labeled_pointclouds = {}

for frames in frame_sets:
    dict_key = f"frames_{'_'.join(map(str, frames))}"
    print(f"\nAccumulating projection for {len(frames)} frames: {frames}")
    
    labeled_pc = accumulate_labels_multi_frame(
        points_df, trajectory_df, frames, rgb_calib, T_Device_Camera, masks_dir, provider, mask_cache=mask_cache
    )
    
    labeled_pointclouds[dict_key] = labeled_pc
    print(f"Completed accumulation for {dict_key}")

Pre-loading SAM2 masks to optimize execution...

Accumulating projection for 3 frames: [50, 100, 250]


Projecting points: 100%|██████████| 239524/239524 [00:18<00:00, 12722.33it/s]


Completed accumulation for frames_50_100_250

Accumulating projection for 5 frames: [50, 100, 250, 350, 480]


Projecting points: 100%|██████████| 239524/239524 [00:19<00:00, 12398.20it/s]


Completed accumulation for frames_50_100_250_350_480

Accumulating projection for 7 frames: [50, 100, 250, 350, 480, 150, 200]


Projecting points: 100%|██████████| 239524/239524 [00:20<00:00, 11852.18it/s]


Completed accumulation for frames_50_100_250_350_480_150_200


### 8.4.2 Save Multi-Frame Results as .ply
We export the accumulated pointclouds for each frame set. These files contain the full geometry and the updated semantic colors, ready for quantitative evaluation.

In [35]:
import os
import numpy as np
from plyfile import PlyData, PlyElement

# Output directory for the accumulated pointclouds
pcd_dir = os.path.join('..', 'data', 'outputs', 'segmentation', 'kettle_and_forklift', 'pointclouds', 'multi_frame_comparison')
os.makedirs(pcd_dir, exist_ok=True)

# Color mapping (0 = Grey, 1 = Red, 2 = Cyan)
save_label_to_color = {0: [180, 180, 180], 1: [255, 0, 0], 2: [0, 255, 255]}

for key, labeled_pc in labeled_pointclouds.items():
    print(f"Exporting pointcloud: {key} ...")
    
    all_xyz = labeled_pc[['px_world', 'py_world', 'pz_world']].to_numpy(dtype=np.float32)
    all_labels = labeled_pc['semantic_label'].to_numpy()
    all_colors = np.array([save_label_to_color.get(int(l), [180,180,180]) for l in all_labels], dtype=np.uint8)

    # Compose structured array for the .ply format
    vertex = np.empty(all_xyz.shape[0], dtype=[('x', 'f4'), ('y', 'f4'), ('z', 'f4'), ('red', 'u1'), ('green', 'u1'), ('blue', 'u1'), ('label', 'u1')])
    vertex['x'] = all_xyz[:, 0]
    vertex['y'] = all_xyz[:, 1]
    vertex['z'] = all_xyz[:, 2]
    vertex['red'] = all_colors[:, 0]
    vertex['green'] = all_colors[:, 1]
    vertex['blue'] = all_colors[:, 2]
    vertex['label'] = all_labels.astype(np.uint8)

    # Write to disk
    ply = PlyData([PlyElement.describe(vertex, 'vertex')], text=True)
    out_path = os.path.join(pcd_dir, f"pointcloud_{key}.ply")
    ply.write(out_path)
    
    print(f"Saved: {out_path}")

Exporting pointcloud: frames_50_100_250 ...
Saved: ..\data\outputs\segmentation\kettle_and_forklift\pointclouds\multi_frame_comparison\pointcloud_frames_frames_50_100_250.ply
Exporting pointcloud: frames_50_100_250_350_480 ...
Saved: ..\data\outputs\segmentation\kettle_and_forklift\pointclouds\multi_frame_comparison\pointcloud_frames_frames_50_100_250_350_480.ply
Exporting pointcloud: frames_50_100_250_350_480_150_200 ...
Saved: ..\data\outputs\segmentation\kettle_and_forklift\pointclouds\multi_frame_comparison\pointcloud_frames_frames_50_100_250_350_480_150_200.ply


### 8.4.3 Visual Comparison: 1 Frame vs 7 Frames
We use Rerun to qualitatively compare the baseline projection (1 frame) against our maximum accumulation set (7 frames). This highlights how multi-frame projection resolves occlusion issues and fills in the 3D geometry of the target objects.

In [37]:
import rerun as rr
import numpy as np

# Get Baseline (1 Frame) and Accumulated (7 Frames) 
baseline_df = labeled_baseline_df
accum_df = labeled_pointclouds['frames_50_100_250_350_480_150_200']

# Filter Valid Points Only 
baseline_valid = baseline_df[baseline_df['semantic_label'] > 0]
accum_valid = accum_df[accum_df['semantic_label'] > 0]

baseline_xyz = baseline_valid[['px_world', 'py_world', 'pz_world']].to_numpy(dtype=np.float32)
baseline_labels = baseline_valid['semantic_label'].to_numpy()

accum_xyz = accum_valid[['px_world', 'py_world', 'pz_world']].to_numpy(dtype=np.float32)
accum_labels = accum_valid['semantic_label'].to_numpy()

# Color Mapping
vis_label_to_color = {1: [255, 0, 0], 2: [0, 255, 255]}
baseline_colors = np.array([vis_label_to_color.get(int(l), [0, 255, 0]) for l in baseline_labels], dtype=np.uint8)
accum_colors = np.array([vis_label_to_color.get(int(l), [0, 255, 0]) for l in accum_labels], dtype=np.uint8)

# Center World 
traj_xyz = trajectory_df[['tx_world_device', 'ty_world_device', 'tz_world_device']].to_numpy(dtype=np.float32)
center = np.mean(traj_xyz, axis=0, keepdims=True)

# Rerun Visualization
rr.init("Aria_MultiFrame_Comparison", spawn=True)

rr.log("world/trajectory", rr.LineStrips3D([traj_xyz - center], colors=[[0,255,0]]))

# Log Baseline 
rr.log("world/pointcloud/1_frame_baseline", rr.Points3D(baseline_xyz - center, colors=baseline_colors, radii=0.0005))

# Log Accumulated 
rr.log("world/pointcloud/7_frames_accumulated", rr.Points3D(accum_xyz - center, colors=accum_colors, radii=0.0005))

print(f"Sent comparison to Rerun:")
print(f"- Baseline (1 Frame): {len(baseline_xyz)} points")
print(f"- Accumulated (7 Frames): {len(accum_xyz)} points")
print("Toggle visibility in the Rerun UI to compare the coverage.")

RuntimeError: gRPC connection gracefully disconnected, uri: rerun+http://127.0.0.1:9876/proxy

### Multi frame Comparison conlusion
Based on the visual and quantitative comparison, the pointcloud obtained by accumulating labels from 3 frames (50, 100, 250) provides the best trade-off between correct semantic labeling and the presence of outliers. This configuration maximizes the number of correctly labeled points on the target objects while minimizing the inclusion of spurious or misprojected points. Adding more frames increases coverage but also introduces more outliers, so the 3-frame result is recommended for downstream tasks and visualization and will be used for the final comparison with multi-camera projection.